# 00 — Neo4j Database Exploration
**No API calls required.** Uses only the existing Neo4j AuraDB data.

We have 3 artists, 490 tracks, 91 albums, 8 production companies already ingested.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from dotenv import load_dotenv
from neo4j import GraphDatabase

load_dotenv('../.env')

FIGURES = Path('../paper/figures')
FIGURES.mkdir(parents=True, exist_ok=True)
PROCESSED = Path('../data/processed')
PROCESSED.mkdir(parents=True, exist_ok=True)

driver = GraphDatabase.driver(
    os.getenv('NEO4J_URI'),
    auth=(os.getenv('NEO4J_USER'), os.getenv('NEO4J_PASSWORD'))
)

def run_query(cypher, **params):
    with driver.session() as s:
        result = s.run(cypher, **params)
        return [dict(r) for r in result]

# Quick connection test
test = run_query('RETURN 1 AS ok')
print('Neo4j connected OK' if test else 'Connection failed')

## 1. Node & Relationship Counts

In [ ]:
print('=== Node counts ===')
for label in ['Artist', 'Track', 'Album', 'Playlist', 'ProductionCompany']:
    result = run_query(f'MATCH (n:{label}) RETURN count(n) AS c')
    print(f'  {label:<20} {result[0]["c"]:>6}')

print()
print('=== Relationship counts ===')
rels = run_query('MATCH ()-[r]->() RETURN type(r) AS t, count(r) AS c ORDER BY c DESC')
for r in rels:
    print(f'  {r["t"]:<25} {r["c"]:>6}')

## 2. Per-Artist Detail

In [ ]:
artists = run_query('MATCH (a:Artist) RETURN a.spotify_id AS id, a.name AS name, a.is_ghost AS is_ghost, a.label AS label')
print(f'Artists in DB: {len(artists)}')

for artist in artists:
    aid = artist['id']
    name = artist['name']
    print(f'\n--- {name} ({"GHOST" if artist["is_ghost"] else "organic"}) ---')

    # Album count
    albums = run_query(
        'MATCH (a:Artist {spotify_id:$id})-[:RELEASED]->(al:Album) RETURN count(al) AS c', id=aid
    )
    print(f'  Albums: {albums[0]["c"]}')

    # Track count
    tracks = run_query(
        'MATCH (a:Artist {spotify_id:$id})-[:RELEASED]->(al:Album)-[:CONTAINS]->(t:Track) RETURN count(t) AS c', id=aid
    )
    print(f'  Tracks: {tracks[0]["c"]}')

    # ISRC prefixes
    isrcs = run_query(
        '''MATCH (a:Artist {spotify_id:$id})-[:RELEASED]->(al:Album)-[:CONTAINS]->(t:Track)
           WHERE t.isrc <> ""
           RETURN t.isrc AS isrc LIMIT 20''', id=aid
    )
    if isrcs:
        prefixes = set(r['isrc'][:5] for r in isrcs if r['isrc'])
        print(f'  ISRC prefixes: {sorted(prefixes)}')
    else:
        print('  ISRC prefixes: none found')

    # Production companies
    companies = run_query(
        '''MATCH (a:Artist {spotify_id:$id})-[:RELEASED]->(al:Album)-[:CONTAINS]->(t:Track)
           -[:REGISTERED_WITH]->(pc:ProductionCompany)
           RETURN DISTINCT pc.name AS company, pc.isrc_prefix AS prefix''', id=aid
    )
    if companies:
        for c in companies:
            print(f'  Production company: {c["company"]} ({c["prefix"]})')
    else:
        print('  Production companies: none linked')

## 3. ISRC Prefix Distribution

In [ ]:
isrc_data = run_query(
    '''MATCH (t:Track) WHERE t.isrc <> "" AND t.isrc IS NOT NULL
       RETURN t.isrc AS isrc'''
)

if isrc_data:
    from collections import Counter
    prefixes = [r['isrc'][:5] for r in isrc_data]
    prefix_counts = Counter(prefixes)
    prefix_df = pd.DataFrame(prefix_counts.most_common(20), columns=['prefix', 'count'])

    print('Top 20 ISRC prefixes in DB:')
    print(prefix_df.to_string(index=False))

    fig, ax = plt.subplots(figsize=(10, 6))
    prefix_df.sort_values('count').plot(kind='barh', x='prefix', y='count', ax=ax,
                                         color='#9b59b6', alpha=0.8, legend=False)
    ax.set_xlabel('Track Count')
    ax.set_title('ISRC Prefix Distribution in Neo4j\n(Each prefix = one production company/country)',
                 fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.savefig(FIGURES / 'neo4j_isrc_prefixes.png', dpi=300, bbox_inches='tight', facecolor='white')
    plt.show()
    print('Saved: neo4j_isrc_prefixes.png')
else:
    print('No ISRC data found in tracks')

## 4. Production Company Overlap (Exercise 3 Preview)

In [ ]:
# Which production companies are shared across multiple artists?
shared = run_query(
    '''MATCH (a:Artist)-[:RELEASED]->(al:Album)-[:CONTAINS]->(t:Track)
       -[:REGISTERED_WITH]->(pc:ProductionCompany)
       WITH pc, collect(DISTINCT a.name) AS artists, count(DISTINCT a) AS artist_count
       RETURN pc.name AS company, pc.isrc_prefix AS prefix,
              artist_count, artists
       ORDER BY artist_count DESC'''
)

if shared:
    print('Production company → artist connections:')
    for row in shared:
        flag = '*** SHARED ***' if row['artist_count'] > 1 else ''
        print(f"  [{row['prefix']}] {row['company']}")
        print(f"    Artists ({row['artist_count']}): {row['artists']} {flag}")
else:
    print('No production company → artist links found yet')
    print('(Run the full ingest to populate these)')

## 5. Export Summary

In [ ]:
# Build artist-to-company table
rows = run_query(
    '''MATCH (a:Artist)-[:RELEASED]->(al:Album)-[:CONTAINS]->(t:Track)
       -[:REGISTERED_WITH]->(pc:ProductionCompany)
       RETURN a.name AS artist_name, a.is_ghost AS is_ghost,
              pc.name AS company_name, pc.isrc_prefix AS isrc_prefix,
              count(t) AS track_count
       ORDER BY artist_name, track_count DESC'''
)

summary_df = pd.DataFrame(rows) if rows else pd.DataFrame(
    columns=['artist_name','is_ghost','company_name','isrc_prefix','track_count']
)
summary_df.to_csv(PROCESSED / 'neo4j_summary.csv', index=False)
print(f'Saved neo4j_summary.csv ({len(summary_df)} rows)')
print()
print(summary_df.to_string(index=False) if not summary_df.empty else 'Empty — run full ingest first')

driver.close()
print('\nNeo4j connection closed.')